# Lab 6 Part 1: Makemore - Character-Level Language Model

## 🎯 Learning Objectives

By the end of this lab, you will:
- Understand tokenization and why language models need it
- Build a character-level language model from scratch
- Understand bigram and MLP-based language models
- Train on Shakespeare text and generate text
- Save and load model checkpoints

**What You'll Build:** A Shakespeare text generator using simple MLP architecture

**Dataset:** Tiny Shakespeare (~1MB text from Shakespeare's works)

**Reference:** [Karpathy's makemore](https://github.com/karpathy/makemore)

**Note:** PyTorch is pre-installed in Google Colab!

## Part 0: Setup

Check for GPU, set seeds, and download data if in Colab

In [ ]:
import sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Detect device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("\n✓ GPU detected! Training will be fast.")
else:
    print("\n⚠️  No GPU detected. Training will be slower.")
    print("In Colab: Runtime → Change runtime type → Hardware accelerator → T4 GPU")

In [ ]:
# Download dataset if in Colab
import os

if 'google.colab' in sys.modules:
    if not os.path.exists('shakespeare.txt'):
        print("Downloading shakespeare.txt...")
        !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O shakespeare.txt
        print("✓ Download complete!")
    else:
        print("✓ shakespeare.txt already exists")
else:
    print("Running locally - ensure shakespeare.txt is in current directory")

# Load text
with open('shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"\nDataset size: {len(text):,} characters")
print(f"First 200 characters:\n{text[:200]}")

## Part 1: What is Tokenization?

### The Problem

Neural networks work with numbers, not text. We need to:
- Convert text → numbers (encoding)
- Convert numbers → text (decoding)

### Tokenization Approaches

**1. Character-level** (what we'll use)
- Each character is a token
- Vocabulary: 26 letters + punctuation + space (~100 tokens)
- ✅ Pros: Small vocabulary, no out-of-vocabulary words
- ❌ Cons: Long sequences, harder to learn word meanings

**2. Subword (BPE/WordPiece)** (used by GPT/BERT)
- Tokens are frequent character sequences
- Vocabulary: ~50k tokens ("ing", "the", "tion", etc.)
- ✅ Pros: Efficient, balance between char and word level
- ❌ Cons: Complex algorithm, larger vocabulary

**3. Word-level**
- Each word is a token
- Vocabulary: hundreds of thousands
- ✅ Pros: Natural semantic units
- ❌ Cons: Huge vocabulary, can't handle new words

### Example

```
Text: "Hello world"

Character-level:
tokens: ['H', 'e', 'l', 'l', 'o', ' ', 'w', 'o', 'r', 'l', 'd']
IDs:    [7, 4, 11, 11, 14, 0, 22, 14, 17, 11, 3]

Subword (BPE):
tokens: ['Hello', ' world']
IDs:    [15496, 995]

Word-level:
tokens: ['Hello', 'world']
IDs:    [34521, 1245]
```

### Interactive Tokenizer Visualization

See how text is converted to tokens and back!

In [ ]:
%%html
<style>
#tokenizer-container * { box-sizing: border-box; margin: 0; padding: 0; }
#tokenizer-container { padding: 1rem 0; font-family: sans-serif; color: #1a1a1a; }
#tokenizer-container h1 { font-size: 18px; font-weight: 500; margin-bottom: 4px; }
#tokenizer-container .subtitle { font-size: 13px; color: #666; margin-bottom: 20px; }
#tokenizer-container .section { margin-bottom: 20px; padding: 16px; border: 0.5px solid #ddd; border-radius: 10px; background: #f7f7f5; }
#tokenizer-container .section-label { font-size: 11px; font-weight: 500; color: #666; text-transform: uppercase; letter-spacing: .04em; margin-bottom: 8px; }
#tokenizer-container textarea { width: 100%; padding: 10px; font-family: monospace; font-size: 13px; border: 0.5px solid #ccc; border-radius: 6px; resize: vertical; min-height: 60px; }
#tokenizer-container .token-display { display: flex; flex-wrap: wrap; gap: 6px; min-height: 40px; }
#tokenizer-container .token { padding: 4px 10px; border-radius: 6px; background: #E6F1FB; border: 0.5px solid #185FA5; color: #0C447C; font-size: 12px; font-family: monospace; }
#tokenizer-container .id-display { display: flex; flex-wrap: wrap; gap: 6px; min-height: 40px; }
#tokenizer-container .id { padding: 4px 10px; border-radius: 6px; background: #EEEDFE; border: 0.5px solid #534AB7; color: #3C3489; font-size: 12px; font-family: monospace; }
#tokenizer-container .info { font-size: 12px; color: #666; margin-top: 8px; }
#tokenizer-container .vocab-display { display: grid; grid-template-columns: repeat(auto-fill, minmax(50px, 1fr)); gap: 4px; max-height: 150px; overflow-y: auto; }
#tokenizer-container .vocab-item { padding: 4px 8px; border-radius: 4px; background: #fff; border: 0.5px solid #ddd; font-size: 11px; font-family: monospace; text-align: center; }
</style>

<div id="tokenizer-container">
  <h1>Character-Level Tokenizer</h1>
  <p class="subtitle">Type text and see how it's tokenized for a language model.</p>

  <div class="section">
    <div class="section-label">Input Text</div>
    <textarea id="tok-input" oninput="tokUpdate()">Hello, world!</textarea>
  </div>

  <div class="section">
    <div class="section-label">Step 1: Split into Characters (Tokens)</div>
    <div class="token-display" id="tok-tokens"></div>
    <div class="info" id="tok-token-info"></div>
  </div>

  <div class="section">
    <div class="section-label">Step 2: Map to IDs (Encoding)</div>
    <div class="id-display" id="tok-ids"></div>
    <div class="info" id="tok-id-info"></div>
  </div>

  <div class="section">
    <div class="section-label">Vocabulary (Character → ID)</div>
    <div class="vocab-display" id="tok-vocab"></div>
  </div>
</div>

<script>
(function() {
  const CHARS = ' !,-.?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz';
  const char2id = {};
  const id2char = {};
  
  CHARS.split('').forEach((c, i) => {
    char2id[c] = i;
    id2char[i] = c;
  });

  function renderVocab() {
    const vocabEl = document.getElementById('tok-vocab');
    if (!vocabEl) return;
    vocabEl.innerHTML = CHARS.split('').map((c, i) => {
      const display = c === ' ' ? '␣' : c;
      return `<div class="vocab-item">${display}: ${i}</div>`;
    }).join('');
  }

  window.tokUpdate = function() {
    const input = document.getElementById('tok-input');
    if (!input) return;
    const text = input.value;
    
    const tokens = text.split('');
    const tokensEl = document.getElementById('tok-tokens');
    if (tokensEl) {
      tokensEl.innerHTML = tokens.map(t => {
        const display = t === ' ' ? '␣' : t;
        return `<div class="token">${display}</div>`;
      }).join('');
    }
    
    const tokenInfoEl = document.getElementById('tok-token-info');
    if (tokenInfoEl) {
      tokenInfoEl.textContent = `${tokens.length} tokens`;
    }

    const ids = tokens.map(t => char2id[t] !== undefined ? char2id[t] : '?');
    const idsEl = document.getElementById('tok-ids');
    if (idsEl) {
      idsEl.innerHTML = ids.map(id => {
        return `<div class="id">${id}</div>`;
      }).join('');
    }
    
    const idInfoEl = document.getElementById('tok-id-info');
    if (idInfoEl) {
      const unk = ids.filter(id => id === '?').length;
      if (unk > 0) {
        idInfoEl.textContent = `${ids.length} IDs (⚠️ ${unk} unknown characters)`;
      } else {
        idInfoEl.textContent = `${ids.length} IDs — ready to feed to model!`;
      }
    }
  };

  renderVocab();
  tokUpdate();
})();
</script>

## Exercise 1: Implement Character Tokenizer

### Your Task

Build a character-level tokenizer:
1. Extract unique characters from text (vocabulary)
2. Create mappings: char → ID and ID → char
3. Implement `encode(text)` → list of IDs
4. Implement `decode(ids)` → text

### Hints
- Use `set(text)` to get unique characters
- Use `sorted()` to ensure consistent ordering
- Use dictionary comprehension for mappings
- Use list comprehension for encode/decode

In [ ]:
class CharTokenizer:
    """Simple character-level tokenizer."""
    
    def __init__(self, text):
        # Get unique characters and sort them
        chars = sorted(list(set(text)))
        self.vocab_size = len(chars)
        
        # Create mappings
        self.char2id = {ch: i for i, ch in enumerate(chars)}
        self.id2char = {i: ch for i, ch in enumerate(chars)}
    
    def encode(self, text):
        """Convert text to list of token IDs."""
        return [self.char2id[ch] for ch in text]
    
    def decode(self, ids):
        """Convert list of token IDs to text."""
        return ''.join([self.id2char[i] for i in ids])

# Create tokenizer
tokenizer = CharTokenizer(text)
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"First 20 chars: {''.join(list(tokenizer.id2char.values())[:20])}...")
print(f"\n✓ Tokenizer created!")

### Test Your Tokenizer

In [ ]:
# Test encode
test_text = "Hello"
encoded = tokenizer.encode(test_text)
print(f"Text: '{test_text}'")
print(f"Encoded: {encoded}")

# Test decode
decoded = tokenizer.decode(encoded)
print(f"Decoded: '{decoded}'")

# Verify
assert decoded == test_text, f"Decode failed: {decoded} != {test_text}"
print("\n✓ Tokenizer works!")

## Part 2: Language Modeling Basics

### What is a Language Model?

**Definition:** A model that predicts the next token given previous tokens

**Example:**
```
Input:  "To be or not to"
Output: "be" (predicted next word)
```

**How it works:**
1. Take context (previous tokens)
2. Predict probability distribution over all possible next tokens
3. Sample from distribution to generate next token
4. Repeat to generate text

### N-gram Models

**Bigram:** Predict based on 1 previous token
- P("cat" | "the") = how often "cat" follows "the"
- Simple lookup table
- No long-term memory

**Trigram:** Predict based on 2 previous tokens
- P("cat" | "the", "black") = how often "cat" follows "the black"
- Better context, but table grows exponentially

**Neural N-gram:** Use neural network instead of lookup table
- Can learn patterns and generalize
- This is what we'll build!

### Prepare Training Data

Split into train (90%) and validation (10%)

In [ ]:
# Encode entire dataset
data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
print(f"Encoded dataset shape: {data.shape}")

# Split into train/val
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Train size: {len(train_data):,} tokens")
print(f"Val size: {len(val_data):,} tokens")
print(f"\n✓ Data prepared!")

### Create Data Batches

For training, we need (context, target) pairs

In [ ]:
def get_batch(split, train_data, val_data, batch_size=32, context_length=8, device='cpu'):
    """
    Generate a batch of (context, target) pairs.
    
    Args:
        split: 'train' or 'val'
        batch_size: number of sequences
        context_length: length of each sequence
    Returns:
        x: (batch_size, context_length) context tokens
        y: (batch_size, context_length) target tokens (shifted by 1)
    """
    data = train_data if split == 'train' else val_data
    
    # Random starting positions
    ix = torch.randint(len(data) - context_length, (batch_size,))
    
    # Extract sequences
    x = torch.stack([data[i:i+context_length] for i in ix])
    y = torch.stack([data[i+1:i+context_length+1] for i in ix])
    
    x, y = x.to(device), y.to(device)
    return x, y

# Test batch generation
# xb, yb = get_batch('train', train_data, val_data, batch_size=4, context_length=8, device=device)
# print(f"Input shape: {xb.shape}")
# print(f"Target shape: {yb.shape}")
# print(f"\nExample batch:")
# for i in range(2):
#     context = xb[i].tolist()
#     target = yb[i].tolist()
#     print(f"Context: {tokenizer.decode(context)!r}")
#     print(f"Target:  {tokenizer.decode(target)!r}")
#     print()

## Part 3: Makemore Model Architecture

### Why MLP Instead of Lookup Table?

**Bigram lookup table:** Each token predicts next token independently
- 65 x 65 table for vocab_size=65
- Can't generalize or learn patterns

**Neural MLP:** Learn embeddings and patterns
- Embedding layer: Convert tokens to continuous vectors
- Hidden layers: Learn patterns in embedded space
- Output layer: Predict next token
- Can generalize to unseen combinations

### Architecture

```
Input: [token_1, token_2, ..., token_8] (context_length=8)
  ↓
Embedding: (8, 32) - convert each token to 32-dim vector
  ↓
Flatten: (256,) - flatten all embeddings
  ↓
Linear + ReLU: (256, 128)
  ↓
Linear: (128, vocab_size)
  ↓
Output logits: (vocab_size,) - scores for each possible next token
```

## Exercise 2: Implement Makemore MLP

### Your Task

Build an MLP language model:
1. Embedding layer to convert tokens to vectors
2. Flatten embeddings (batch x context x embed_dim → batch x (context*embed_dim))
3. Hidden layer with ReLU activation
4. Output layer to predict logits

### Hints
- Use `nn.Embedding(vocab_size, embed_dim)`
- Use `view()` or `reshape()` to flatten
- Use `nn.Linear()` for layers
- Use `F.relu()` for activation

In [ ]:
class MakemoreMLP(nn.Module):
    """MLP-based language model."""
    
    def __init__(self, vocab_size, context_length=8, embed_dim=32, hidden_dim=128):
        super().__init__()
        self.context_length = context_length
        self.embed_dim = embed_dim
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # Hidden layer
        self.fc1 = nn.Linear(context_length * embed_dim, hidden_dim)
        
        # Output layer
        self.fc2 = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, idx):
        """
        Args:
            idx: (B, T) tensor of token IDs, T=context_length
        Returns:
            logits: (B, vocab_size) tensor of scores
        """
        B, T = idx.shape
        
        # Get embeddings
        x = self.embedding(idx)  # (B, T, embed_dim)
        
        # Flatten embeddings
        x = x.view(B, T * self.embed_dim)  # (B, T*embed_dim)
        
        # Hidden layer with ReLU
        x = F.relu(self.fc1(x))  # (B, hidden_dim)
        
        # Output layer
        logits = self.fc2(x)  # (B, vocab_size)
        
        return logits

# Create model
model = MakemoreMLP(vocab_size=tokenizer.vocab_size, context_length=8)
model = model.to(device)
print(f"Model created with {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"\n✓ Makemore MLP ready!")

## Exercise 3: Implement Training Loop

### Training Setup

**Loss Function:** Cross-entropy
- Measures how well predicted probabilities match actual next tokens
- Lower is better

**Perplexity:** exp(loss)
- Interpretable metric: "on average, the model is as confused as if it had to choose uniformly from N tokens"
- Example: perplexity of 10 = model is as confused as random choice from 10 options
- Random baseline: perplexity ≈ vocab_size (65 for our tokenizer)

**Expected Results:**
- Loss should decrease from ~4.2 to ~2.0
- Perplexity should decrease from ~65 to ~7-8
- Training takes ~3-5 minutes on GPU, ~15 minutes on CPU

### Your Task

Train the makemore model:
1. Get batch of (context, target) pairs
2. Forward pass: compute logits
3. Compute cross-entropy loss
4. Backward pass: compute gradients
5. Update weights with optimizer
6. Track and print metrics

In [ ]:
# Test batch generation
xb, yb = get_batch('train', train_data, val_data, batch_size=4, context_length=8, device=device)
print(f"Input shape: {xb.shape}")
print(f"Target shape: {yb.shape}")
print(f"\nExample batch:")
for i in range(2):
    context = xb[i].tolist()
    target = yb[i].tolist()
    print(f"Context: {tokenizer.decode(context)!r}")
    print(f"Target:  {tokenizer.decode(target)!r}")
    print()

### Plot Training Loss

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Makemore Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

## Part 4: Evaluation and Generation

### Text Generation

To generate text:
1. Start with a prompt (or random tokens)
2. Use last `context_length` tokens as input
3. Get predicted next token
4. Append to sequence
5. Repeat

### Sampling Strategies

**Greedy:** Always pick highest probability token
- Deterministic but repetitive

**Random sampling:** Sample from probability distribution
- More diverse but potentially incoherent

**Temperature sampling:** Adjust randomness
- temperature < 1.0: More confident (less random)
- temperature > 1.0: More random (more creative)

In [ ]:
# Test generation
model.eval()
prompt = "To be or"
print(f"Prompt: '{prompt}'\n")
print("="*80)
generated = generate(model, tokenizer, prompt, max_new_tokens=200, temperature=1.0, device=device)
print(generated)
print("="*80)

## Part 5: Checkpoint Saving and Loading

### Why Save Checkpoints?

- Training takes time - don't want to retrain from scratch
- Can resume training if interrupted
- Can share trained models

### What to Save?

- Model weights (`model.state_dict()`)
- Optimizer state (for resuming training)
- Hyperparameters (vocab_size, context_length, etc.)
- Training metadata (epoch, loss, etc.)

## Exercise 4: Save and Load Checkpoint

### Your Task

1. Create `checkpoints/` directory
2. Save model checkpoint with metadata
3. Load checkpoint and verify it works

In [ ]:
import os

# Create checkpoints directory
os.makedirs('checkpoints', exist_ok=True)

# Save checkpoint
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'vocab_size': tokenizer.vocab_size,
    'context_length': context_length,
    'embed_dim': 32,
    'hidden_dim': 128,
    'loss': losses[-1],
}
torch.save(checkpoint, 'checkpoints/makemore_model.pt')
print("✓ Checkpoint saved to checkpoints/makemore_model.pt")

# Load checkpoint
checkpoint = torch.load('checkpoints/makemore_model.pt', map_location=device)

# Create new model with saved hyperparameters
loaded_model = MakemoreMLP(
    vocab_size=checkpoint['vocab_size'],
    context_length=checkpoint['context_length'],
    embed_dim=checkpoint['embed_dim'],
    hidden_dim=checkpoint['hidden_dim']
)
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model = loaded_model.to(device)
loaded_model.eval()

print("✓ Checkpoint loaded successfully")

# Test generation with loaded model
generated = generate(loaded_model, tokenizer, "ROMEO:", max_new_tokens=100, device=device)
print(f"\nGenerated text:\n{generated}")

## Part 6: Interactive Predictions

Let's see what the model predicts for different prompts

In [ ]:
# Test predictions
show_top_predictions(model, tokenizer, "To be or", device=device)
show_top_predictions(model, tokenizer, "ROMEO:", device=device)
show_top_predictions(model, tokenizer, "JULIET:", device=device)

## Summary

### What You've Learned

✅ **Tokenization:** Convert text ↔ numbers

✅ **Character-level models:** Simple but limited context

✅ **MLP language models:** Learn embeddings and patterns

✅ **Training:** Cross-entropy loss, perplexity metric

✅ **Generation:** Sampling from probability distributions

✅ **Checkpoints:** Save and load trained models

### Limitations of Makemore

**Fixed context window:** Only 8 characters of context
- Can't remember earlier parts of sentence
- Can't learn long-range dependencies

**No attention:** MLP treats all context tokens equally
- Can't focus on relevant parts
- Can't learn which tokens matter most

**Solution:** Transformers with attention mechanism!

### Next Lab: NanoGPT

In the next lab, we'll build a transformer-based model that:
- Uses attention to focus on relevant context
- Handles longer context (64+ tokens)
- Achieves much better perplexity (~5 vs ~7)
- Generates more coherent text

**Continue to Lab 6 Part 2: NanoGPT!**